
We calibrate using shelter height (**1.6 m**) instead of GCPs. A ray from the shelter base extended 1.6 m vertically must intersect the ray from the apex. If not, `height` or `focal` parameters are incorrect. We estimate parameters across frames using least-squares.

## Key Fixes
- **3D Ray Model:** Replaced 2D horizontal projection with full 3D rays (`dx`, `dy`, azimuth).
- **Pixel-Space Residuals:** Minimizing pixel distance instead of world-space meters eliminates skew from annotation noise.
- **Dynamic Pitch:** `pitch_down` is frame-specific, while `focal_px` is fixed for the lens.
- **Altitude Source:** Uses `height_m` (AGL) instead of `altitude_m` (barometric MSL).


## 1. Constants


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import cv2
import os
import re
import json
from pathlib import Path
from scipy.optimize import least_squares

IMG_WIDTH, IMG_HEIGHT = 1920, 1080
VIDEO_FPS = 60

TELEMETRY_CSV = "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-17-00-04-csv/DJIFlightRecord_2026-04-02_17-00-04.csv"
VIDEO_OFFSET_S = 0.8      # telemetry_time = video_time + VIDEO_OFFSET_S
FRAMES_DIR = "/kaggle/input/datasets/natair/chickens-drone-25k/chickens_drone_25k"
TRACKS_XML = "/kaggle/input/datasets/natair/chicken-trainings-data/tracks_dense.xml"

SHELTER_HEIGHT_M = 1.6    # Both cone- and wedge-shaped shelters are 1.6 m tall

## 2. Telemetry + geodesy

In [ ]:
# Load telemetry and set local origin (REF point). Uses AGL height_m.

telemetry = pd.read_csv(TELEMETRY_CSV)[["time_s", "lat", "lng", "height_m", "yaw_deg"]].dropna().reset_index(drop=True)
REF_LAT, REF_LNG = telemetry["lat"].mean(), telemetry["lng"].median()

# Sanity checks
assert abs(telemetry["lat"].mean() - REF_LAT) < 1e-9
assert abs(telemetry["lng"].median() - REF_LNG) < 1e-9
assert telemetry["lat"].max() - telemetry["lat"].min() < 0.01, \
    f"Suspicious lat spread in telemetry: {telemetry['lat'].max() - telemetry['lat'].min():.5f}°"
assert telemetry["lng"].max() - telemetry["lng"].min() < 0.01, \
    f"Suspicious lng spread in telemetry: {telemetry['lng'].max() - telemetry['lng'].min():.5f}°"

def latlng_to_local_m(lat, lng, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    return (lng - ref_lng) * m_per_deg_lng, (lat - ref_lat) * m_per_deg_lat

def local_m_to_latlng(east, north, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    return ref_lat + north / m_per_deg_lat, ref_lng + east / m_per_deg_lng

def get_drone_state(frame):
    t = frame / VIDEO_FPS + VIDEO_OFFSET_S
    idx = (telemetry["time_s"] - t).abs().idxmin()
    row = telemetry.loc[idx]
    return row["lat"], row["lng"], row["height_m"], row["yaw_deg"]

# Save reference origin
with open("/kaggle/working/ref_point.json", "w") as f:
    json.dump({"REF_LAT": REF_LAT, "REF_LNG": REF_LNG, "telemetry_csv": TELEMETRY_CSV}, f)

print(f"✓ Telemetria uploaded: {len(telemetry)} lines")
print(f"  REF_LAT={REF_LAT:.6f}, REF_LNG={REF_LNG:.6f}")
print(f"  dispersion lat={telemetry['lat'].max()-telemetry['lat'].min():.5f}°, "
      f"lng={telemetry['lng'].max()-telemetry['lng'].min():.5f}°")
print(f"  ref_point.json saved")

Telemetria uploaded: 6233 lines

REF_LAT=52.752677, REF_LNG=13.678908

dispersion lat=0.00051°, lng=0.00065°

ref_point.json saved

## 3. Telemetry outlier check

In [ ]:
# Check for GPS outliers (>1 km from median)
med_lat, med_lng = telemetry["lat"].median(), telemetry["lng"].median()
dist_deg = np.hypot(telemetry["lat"] - med_lat, telemetry["lng"] - med_lng)
outliers = telemetry[dist_deg > 0.01]  # roughly 1km from the median

print(f"median: {med_lat}, {med_lng}")
print(f"suspicious lines: {len(outliers)}")
if len(outliers):
    print(outliers[["time_s", "lat", "lng"]])

median: 52.75268290728269, 13.67890806998152

suspicious lines: 0


## 4. Camera ray model

In [ ]:
# 3D Ray projection and reprojection helper functions
def project_ray_to_height(drone_east, drone_north, height, yaw_deg, pitch_down_deg, focal_px,
                           px, py, target_height=0.0):
    """Projects pixel (px, py) to a ground height plane, returning (East, North) meters."""
    yaw, pitch = np.radians(yaw_deg), np.radians(pitch_down_deg)
    forward = np.array([np.sin(yaw)*np.cos(pitch), np.cos(yaw)*np.cos(pitch), -np.sin(pitch)])
    right = np.array([np.cos(yaw), -np.sin(yaw), 0])
    down = np.cross(forward, right)
    dx, dy = px - IMG_WIDTH/2, py - IMG_HEIGHT/2
    d = right*(dx/focal_px) + down*(dy/focal_px) + forward
    d = d / np.linalg.norm(d)
    if d[2] >= -1e-6:
        return None   # Ray points upward
    t = (target_height - height) / d[2]
    return drone_east + t*d[0], drone_north + t*d[1]

def world_to_pixel(drone_east, drone_north, height, yaw_deg, pitch_down_deg, focal_px, X, Y, Z):
    """Reprojects 3D point (X, Y, Z) back to image pixel space (px, py)."""
    yaw, pitch = np.radians(yaw_deg), np.radians(pitch_down_deg)
    forward = np.array([np.sin(yaw)*np.cos(pitch), np.cos(yaw)*np.cos(pitch), -np.sin(pitch)])
    right = np.array([np.cos(yaw), -np.sin(yaw), 0])
    down = np.cross(forward, right)
    rel = np.array([X-drone_east, Y-drone_north, Z-height])
    lf, lr, ld = rel@forward, rel@right, rel@down
    if lf <= 1e-6:
        return None   # point is behind the camera
    return IMG_WIDTH/2 + focal_px*lr/lf, IMG_HEIGHT/2 + focal_px*ld/lf

Frames found: 25338

## 5. Frame browsing helpers

In [ ]:
# Helpers to browse frames: contact_sheet() for batch scan, show_grid() for precise pixels

def frame_number(path):
    match = re.search(r'(\d+)', path.stem)
    return int(match.group(1)) if match else 0

EXTS = {'.jpg', '.jpeg', '.png'}
frame_files = sorted(
    [p for p in Path(FRAMES_DIR).rglob("*") if p.suffix.lower() in EXTS],
    key=frame_number
)
print(f"✓ Frames found: {len(frame_files)}")

def contact_sheet(frame_indices, cols=5):
    rows = (len(frame_indices) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows))
    axes = axes.flat if rows > 1 else ([axes] if cols == 1 else axes)
    for ax, idx in zip(axes, frame_indices):
        img = cv2.cvtColor(cv2.imread(str(frame_files[idx])), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"Frame {idx}", fontsize=9)
        ax.axis("off")
    for ax in list(axes)[len(frame_indices):]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig("/kaggle/working/contact_sheet.jpg", dpi=100, bbox_inches="tight")
    plt.show()

def show_grid(frame_idx, step=100):
    fp = frame_files[frame_idx]
    img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    fig, ax = plt.subplots(figsize=(18, 10))
    ax.imshow(img)
    for x in range(0, W, step):
        ax.axvline(x, color='yellow', alpha=0.25, linewidth=0.5)
        ax.text(x, 15, str(x), color='red', fontsize=7)
    for y in range(0, H, step):
        ax.axhline(y, color='yellow', alpha=0.25, linewidth=0.5)
        ax.text(5, y, str(y), color='red', fontsize=7)
    ax.set_title(f"Frame {frame_idx}")
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/grid_frame_{frame_idx}.jpg", dpi=100, bbox_inches="tight")
    plt.show()

# Example - a wide contact sheet first, to spot frames with visible shelters:
# contact_sheet(list(range(0, 5000, 300)))
# then a precise grid view of one:
# show_grid(1500)

## 6. Click-to-coordinates tool

In [ ]:
# Interactive HTML/JS tool to log image pixel coordinates via clicks

import base64
from IPython.display import HTML

def click_to_coords(frame_idx):
    img_path = str(frame_files[frame_idx])
    with open(img_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()

    html = f"""
    <div style="font-family: monospace;">
      <div style="margin-bottom:8px; font-size:14px;">Frame {frame_idx} - click</div>
      <img id="clkimg_{frame_idx}" src="data:image/jpeg;base64,{img_b64}"
           style="max-width:100%; cursor:crosshair; border:1px solid #888;" />
      <div id="log_{frame_idx}" style="margin-top:10px; font-size:14px; line-height:1.6;"></div>
      <button id="reset_{frame_idx}" style="margin-top:8px; padding:4px 12px; cursor:pointer;">
        Clear
      </button>
    </div>
    <script>
    (function() {{
        var img = document.getElementById('clkimg_{frame_idx}');
        var log = document.getElementById('log_{frame_idx}');
        var resetBtn = document.getElementById('reset_{frame_idx}');
        var clicks = [];

        img.onclick = function(e) {{
            var rect = img.getBoundingClientRect();
            var scaleX = img.naturalWidth / rect.width;
            var scaleY = img.naturalHeight / rect.height;
            var x = Math.round((e.clientX - rect.left) * scaleX);
            var y = Math.round((e.clientY - rect.top) * scaleY);
            clicks.push([x, y]);
            renderLog();
        }};

        resetBtn.onclick = function() {{ clicks = []; renderLog(); }};

        function renderLog() {{
            var lines = clicks.map(function(c, i) {{
                return "Точка " + (i+1) + ":  px=" + c[0] + ",  py=" + c[1];
            }});
            log.innerHTML = lines.join("<br>");
        }}
    }})();
    </script>
    """
    return HTML(html)

click_to_coords(25300)

![Frame 25300 - click](../../data/images/Screenshot%202026-09-15%20at%2014.29.48.png)

Frame 1: px=506, py=769

Frame 2: px=1027, py=746

Frame 3: px=1365, py=787

Frame 4: px=1467, py=859

## 7. Shelter sightings + calibration

In [ ]:
# Base/top pixel pairs for the same shelter, same frame. No real-world position needed.

SHELTER_SIGHTINGS = [
    {"frame": 1, "base_px": 1265, "base_py": 567, "top_px": 1262, "top_py": 470},
    {"frame": 1, "base_px": 719, "base_py": 534, "top_px": 725, "top_py": 460},
    {"frame": 100, "base_px": 1129, "base_py": 524, "top_px": 1128, "top_py": 449},
    {"frame": 100, "base_px": 1715, "base_py": 573, "top_px": 1715, "top_py": 469},
    {"frame": 200, "base_px": 1827, "base_py": 680, "top_px": 1827, "top_py": 556},
    {"frame": 200, "base_px": 1210, "base_py": 617, "top_px": 1215, "top_py": 527},
    {"frame": 300, "base_px": 1098, "base_py": 677, "top_px": 1098, "top_py": 586},
    {"frame": 300, "base_px": 1684, "base_py": 758, "top_px": 1684, "top_py": 640},
    {"frame": 400, "base_px": 1034, "base_py": 606, "top_px": 1034, "top_py": 504},
    {"frame": 400, "base_px": 490, "base_py": 534, "top_px": 490, "top_py": 447},
    {"frame": 500, "base_px": 1088, "base_py": 696, "top_px": 1088, "top_py": 596},
    {"frame": 500, "base_px": 554, "base_py": 606, "top_px": 554, "top_py": 524},
    {"frame": 600, "base_px": 1108, "base_py": 698, "top_px": 1108, "top_py": 599},
    {"frame": 600, "base_px": 584, "base_py": 582, "top_px": 584, "top_py": 501},
    {"frame": 600, "base_px": 303, "base_py": 428, "top_px": 303, "top_py": 373},
    {"frame": 600, "base_px": 225, "base_py": 394, "top_px": 225, "top_py": 347},
    {"frame": 700, "base_px": 1111, "base_py": 733, "top_px": 1111, "top_py": 631},
    {"frame": 700, "base_px": 597, "base_py": 593, "top_px": 597, "top_py": 513},
    {"frame": 700, "base_px": 315, "base_py": 406, "top_px": 315, "top_py": 351},
    {"frame": 700, "base_px": 237, "base_py": 362, "top_px": 237, "top_py": 316},
    {"frame": 800, "base_px": 1094, "base_py": 667, "top_px": 1094, "top_py": 579},
    {"frame": 800, "base_px": 591, "base_py": 518, "top_px": 591, "top_py": 444},
    {"frame": 800, "base_px": 301, "base_py": 292, "top_px": 301, "top_py": 240},
    {"frame": 800, "base_px": 219, "base_py": 249, "top_px": 219, "top_py": 200},
    {"frame": 900, "base_px": 1098, "base_py": 774, "top_px": 1098, "top_py": 693},
    {"frame": 900, "base_px": 607, "base_py": 617, "top_px": 607, "top_py": 545},
    {"frame": 900, "base_px": 324, "base_py": 365, "top_px": 324, "top_py": 316},
    {"frame": 900, "base_px": 243, "base_py": 318, "top_px": 243, "top_py": 272},
    {"frame": 1000, "base_px": 1060, "base_py": 715, "top_px": 1060, "top_py": 635},
    {"frame": 1000, "base_px": 580, "base_py": 550, "top_px": 580, "top_py": 479},
    {"frame": 1000, "base_px": 288, "base_py": 272, "top_px": 288, "top_py": 218},
    {"frame": 1000, "base_px": 205, "base_py": 217, "top_px": 205, "top_py": 166},
    {"frame": 1100, "base_px": 1071, "base_py": 756, "top_px": 1071, "top_py": 681},
    {"frame": 1100, "base_px": 609, "base_py": 580, "top_px": 609, "top_py": 512},
    {"frame": 1100, "base_px": 320, "base_py": 276, "top_px": 320, "top_py": 226},
    {"frame": 1100, "base_px": 236, "base_py": 220, "top_px": 236, "top_py": 171},
    {"frame": 1500, "base_px": 1374, "base_py": 1039, "top_px": 1374, "top_py": 964},
    {"frame": 1500, "base_px": 971, "base_py": 738, "top_px": 971, "top_py": 672},
    {"frame": 1500, "base_px": 815, "base_py": 379, "top_px": 815, "top_py": 333},
    {"frame": 1500, "base_px": 762, "base_py": 310, "top_px": 762, "top_py": 267},
    {"frame": 1800, "base_px": 1095, "base_py": 904, "top_px": 1095, "top_py": 834},
    {"frame": 1800, "base_px": 883, "base_py": 623, "top_px": 883, "top_py": 564},
    {"frame": 1800, "base_px": 915, "base_py": 324, "top_px": 915, "top_py": 284},
    {"frame": 1800, "base_px": 900, "base_py": 263, "top_px": 900, "top_py": 227},
    {"frame": 2100, "base_px": 1114, "base_py": 677, "top_px": 1114, "top_py": 605},
    {"frame": 2100, "base_px": 1579, "base_py": 383, "top_px": 1579, "top_py": 331},
    {"frame": 2100, "base_px": 1639, "base_py": 316, "top_px": 1639, "top_py": 269},
    {"frame": 2500, "base_px": 546, "base_py": 768, "top_px": 546, "top_py": 675},
    {"frame": 2500, "base_px": 1072, "base_py": 550, "top_px": 1072, "top_py": 469},
    {"frame": 2800, "base_px": 165, "base_py": 1004, "top_px": 165, "top_py": 894},
    {"frame": 2800, "base_px": 1167, "base_py": 483, "top_px": 1167, "top_py": 383},
    {"frame": 3100, "base_px": 1213, "base_py": 1022, "top_px": 1245, "top_py": 942},
    {"frame": 3500, "base_px": 1210, "base_py": 921, "top_px": 1245, "top_py": 828},
    {"frame": 3700, "base_px": 1232, "base_py": 898, "top_px": 1232, "top_py": 814},
    {"frame": 4150, "base_px": 1531, "base_py": 255, "top_px": 1567, "top_py": 152},
    {"frame": 4500, "base_px": 968, "base_py": 233, "top_px": 976, "top_py": 125},
    {"frame": 4800, "base_px": 776, "base_py": 434, "top_px": 774, "top_py": 333},
    {"frame": 5100, "base_px": 1354, "base_py": 458, "top_px": 1331, "top_py": 354},
    {"frame": 5500, "base_px": 1513, "base_py": 466, "top_px": 1534, "top_py": 357},
    {"frame": 6200, "base_px": 1037, "base_py": 623, "top_px": 1054, "top_py": 510},
    {"frame": 6700, "base_px": 381, "base_py": 866, "top_px": 337, "top_py": 761},
    {"frame": 6850, "base_px": 404, "base_py": 912, "top_px": 381, "top_py": 813},
    {"frame": 9800, "base_px": 1562, "base_py": 1060, "top_px": 1635, "top_py": 1054},
    {"frame": 10000, "base_px": 1005, "base_py": 828, "top_px": 1019, "top_py": 718},
    {"frame": 10500, "base_px": 681, "base_py": 767, "top_px": 681, "top_py": 652},
    {"frame": 11000, "base_px": 678, "base_py": 785, "top_px": 678, "top_py": 663},
    {"frame": 11500, "base_px": 999, "base_py": 799, "top_px": 999, "top_py": 687},
    {"frame": 11700, "base_px": 451, "base_py": 522, "top_px": 451, "top_py": 400},
    {"frame": 11800, "base_px": 370, "base_py": 706, "top_px": 370, "top_py": 608},
    {"frame": 12500, "base_px": 818, "base_py": 447, "top_px": 818, "top_py": 340},
    {"frame": 12800, "base_px": 942, "base_py": 360, "top_px": 942, "top_py": 233},
    {"frame": 13100, "base_px": 942, "base_py": 661, "top_px": 942, "top_py": 493},
    {"frame": 21700, "base_px": 1152, "base_py": 1005, "top_px": 1169, "top_py": 939},
    {"frame": 22000, "base_px": 1141, "base_py": 322, "top_px": 1155, "top_py": 233},
    {"frame": 22000, "base_px": 386, "base_py": 470, "top_px": 373, "top_py": 385},
    {"frame": 22500, "base_px": 1135, "base_py": 304, "top_px": 1149, "top_py": 223},
    {"frame": 22500, "base_px": 378, "base_py": 454, "top_px": 366, "top_py": 373},
    {"frame": 23000, "base_px": 1135, "base_py": 305, "top_px": 1146, "top_py": 220},
    {"frame": 23000, "base_px": 372, "base_py": 460, "top_px": 364, "top_py": 373},
    {"frame": 23500, "base_px": 1008, "base_py": 298, "top_px": 1008, "top_py": 217},
    {"frame": 23500, "base_px": 217, "base_py": 454, "top_px": 194, "top_py": 368},
    {"frame": 24000, "base_px": 555, "base_py": 597, "top_px": 549, "top_py": 513},
    {"frame": 24000, "base_px": 1296, "base_py": 660, "top_px": 1314, "top_py": 585},
    {"frame": 24500, "base_px": 545, "base_py": 483, "top_px": 545, "top_py": 398},
    {"frame": 24500, "base_px": 1305, "base_py": 548, "top_px": 1305, "top_py": 467},
    {"frame": 25000, "base_px": 548, "base_py": 498, "top_px": 543, "top_py": 415},
    {"frame": 25000, "base_px": 1297, "base_py": 567, "top_px": 1316, "top_py": 481},
    {"frame": 25300, "base_px": 1053, "base_py": 640, "top_px": 1062, "top_py": 556},
    {"frame": 25300, "base_px": 194, "base_py": 725, "top_px": 168, "top_py": 631},
]

# Attach drone state (per-observation frame) to each sighting.
for s in SHELTER_SIGHTINGS:
    lat, lng, h, yaw = get_drone_state(s["frame"])
    s["height_m"], s["yaw_deg"] = h, yaw

# Drop takeoff/landing observations (height ≈ 0) - geometry is degenerate.
MIN_CALIB_HEIGHT_M = 5.0
dropped = [s for s in SHELTER_SIGHTINGS if s["height_m"] < MIN_CALIB_HEIGHT_M]
if dropped:
    dropped_frames = sorted(set(s["frame"] for s in dropped))
    print(f"⚠ Left behind {len(dropped)} frames from height < {MIN_CALIB_HEIGHT_M} м "
          f"(takeoff, geometry is degenerate): frames {dropped_frames}")
SHELTER_SIGHTINGS = [s for s in SHELTER_SIGHTINGS if s["height_m"] >= MIN_CALIB_HEIGHT_M]
print(f"Left {len(SHELTER_SIGHTINGS)} observations for calibration")

Left behind 12 frames from height < 5.0 м (takeoff, geometry is degenerate): frames [1, 100, 200, 300, 400, 500]

Left 77 observations for calibration

In [ ]:
# Calibration: one shared focal length for the video; pitch fit per-frame (gimbal isn't constant). Residual in pixels: reprojected top vs. marked top.

unique_frames = sorted(set(s["frame"] for s in SHELTER_SIGHTINGS))
frame_to_idx = {f: i for i, f in enumerate(unique_frames)}

def calib_residuals(params, obs):
    focal = params[0]
    pitches = params[1:]
    errs = []
    for o in obs:
        pitch = pitches[frame_to_idx[o["frame"]]]
        base_xy = project_ray_to_height(0, 0, o["height_m"], o["yaw_deg"], pitch, focal,
                                         o["base_px"], o["base_py"], 0)
        if base_xy is None:
            errs.extend([80, 80]); continue
        pred_top = world_to_pixel(0, 0, o["height_m"], o["yaw_deg"], pitch, focal,
                                   base_xy[0], base_xy[1], SHELTER_HEIGHT_M)
        if pred_top is None:
            errs.extend([80, 80]); continue
        errs.extend([pred_top[0]-o["top_px"], pred_top[1]-o["top_py"]])
    return errs

n_pitch = len(unique_frames)
result = least_squares(calib_residuals, x0=[1500.0]+[45.0]*n_pitch,
                        bounds=([300]+[5]*n_pitch, [4000]+[89]*n_pitch),
                        max_nfev=40000, args=(SHELTER_SIGHTINGS,))
FOCAL_PX = result.x[0]
pitches = result.x[1:]
PITCH_BY_FRAME = dict(zip(unique_frames, pitches))

print(f"focal_px (general, free) = {FOCAL_PX:.0f}px")
print(f"pitch_down: from {pitches.min():.1f}° to {pitches.max():.1f}°  (median {np.median(pitches):.1f}°)")

# Per-observation error - worst cases flag frames worth re-marking.
diag = []
for o in SHELTER_SIGHTINGS:
    pitch = PITCH_BY_FRAME[o["frame"]]
    base_xy = project_ray_to_height(0, 0, o["height_m"], o["yaw_deg"], pitch, FOCAL_PX,
                                     o["base_px"], o["base_py"], 0)
    pred_top = world_to_pixel(0, 0, o["height_m"], o["yaw_deg"], pitch, FOCAL_PX,
                               base_xy[0], base_xy[1], SHELTER_HEIGHT_M) if base_xy else None
    err = np.hypot(pred_top[0]-o["top_px"], pred_top[1]-o["top_py"]) if pred_top else float("inf")
    diag.append((o["frame"], err))

diag.sort(key=lambda x: -x[1])
print(f"\n{'frame':>7}  {'error,px':>10}")
for f, e in diag[:20]:
    print(f"{f:>7}  {e:>10.1f}  {'⚠' if e > 40 else ''}")
print(f"... (the 20 worst ones are shown {len(diag)})")
print(f"\nmedian of error: {np.median([e for _,e in diag]):.1f}px")

# Flag implausible pitch jumps between adjacent frames - usually a labeling error, not real gimbal motion.
sorted_frames = sorted(PITCH_BY_FRAME.keys())
print("\nPitch jumps between adjacent calibrated frames (>15°/s):")
for f1, f2 in zip(sorted_frames, sorted_frames[1:]):
    dp = abs(PITCH_BY_FRAME[f2] - PITCH_BY_FRAME[f1])
    dt = (f2 - f1) / VIDEO_FPS
    if dp / max(dt, 0.1) > 15:
        print(f"  ⚠ {f1}→{f2} ({dt:.1f}s): {PITCH_BY_FRAME[f1]:.1f}°→{PITCH_BY_FRAME[f2]:.1f}° "
              f"({dp/max(dt,0.1):.1f}°/s) - need double-check the labeling of these frames.")

focal_px (general, free) = 2385px
pitch_down: from 8.2° to 76.6°  (median 26.1°)

  frame    error,px
   5100        31.4  
   2800        21.8  
   4150        19.4  
  11700        16.9  
   6850        16.6  
   6200        15.2  
  24000        14.5  
   2800        14.2  
   3500        12.2  
  25000        11.9  
   6700        11.5  
  22500        11.0  
  22000        10.6  
  10000        10.1  
  25300         9.2  
  11800         9.2  
  24500         8.9  
  11000         8.1  
    600         8.1  
   3100         8.0  
... (the 20 worst ones are shown 77)

median of error: 5.0px

Pitch jumps between adjacent calibrated frames (>15°/s):
  6700→6850 (2.5s): 58.3°→15.3° (17.2°/s) - need double-check the labeling of these frames.

## 8. Pitch interpolation + trust cutoffs

In [ ]:
# Interpolate pitch linearly between calibrated frames. Below min pitch/height, drop the detection instead of keeping an unreliable estimate.

_pitch_frames = np.array(sorted(PITCH_BY_FRAME.keys()))
_pitch_values = np.array([PITCH_BY_FRAME[f] for f in _pitch_frames])

MIN_PITCH_FOR_TRUST_DEG = 25.0
MIN_HEIGHT_FOR_TRUST_M = 8.0

def get_pitch_for_frame(frame):
    pitch = np.interp(frame, _pitch_frames, _pitch_values)
    gap = np.min(np.abs(_pitch_frames - frame))   # distance to nearest calibrated frame
    return pitch, gap

def pixel_to_latlng(px, py, frame):
    lat, lng, h, yaw = get_drone_state(frame)
    if h < MIN_HEIGHT_FOR_TRUST_M:
        return None
    de, dn = latlng_to_local_m(lat, lng, REF_LAT, REF_LNG)
    pitch, gap = get_pitch_for_frame(frame)
    if pitch < MIN_PITCH_FOR_TRUST_DEG:
        return None
    proj = project_ray_to_height(de, dn, h, yaw, pitch, FOCAL_PX, px, py, target_height=0)
    if proj is None:
        return None
    lat_out, lon_out = local_m_to_latlng(proj[0], proj[1], REF_LAT, REF_LNG)
    return lat_out, lon_out, gap

## 9. Optional: offset search

In [ ]:
# Sweep VIDEO_OFFSET_S, refit pitch (focal fixed), pick lowest median error. Data-driven check on video/telemetry sync.

def calibration_error_for_offset(offset_s, sightings_raw, focal_px, min_calib_height_m=5.0):
    sightings = [dict(s) for s in sightings_raw]
    old_offset = globals()["VIDEO_OFFSET_S"]
    globals()["VIDEO_OFFSET_S"] = offset_s
    try:
        for s in sightings:
            lat, lng, h, yaw = get_drone_state(s["frame"])
            s["height_m"], s["yaw_deg"] = h, yaw
        sightings = [s for s in sightings if s["height_m"] >= min_calib_height_m]
        if len(sightings) < 10:
            return None, len(sightings)

        unique_frames = sorted(set(s["frame"] for s in sightings))
        frame_to_idx = {f: i for i, f in enumerate(unique_frames)}

        def residuals(pitches):
            errs = []
            for o in sightings:
                pitch = pitches[frame_to_idx[o["frame"]]]
                base_xy = project_ray_to_height(0, 0, o["height_m"], o["yaw_deg"], pitch, focal_px,
                                                 o["base_px"], o["base_py"], 0)
                if base_xy is None:
                    errs.extend([80, 80]); continue
                pred_top = world_to_pixel(0, 0, o["height_m"], o["yaw_deg"], pitch, focal_px,
                                           base_xy[0], base_xy[1], SHELTER_HEIGHT_M)
                if pred_top is None:
                    errs.extend([80, 80]); continue
                errs.extend([pred_top[0]-o["top_px"], pred_top[1]-o["top_py"]])
            return errs

        n = len(unique_frames)
        result = least_squares(residuals, x0=[45.0]*n, bounds=([5]*n, [89]*n), max_nfev=20000)
        errs = np.array(residuals(result.x)).reshape(-1, 2)
        med_err = np.median(np.hypot(errs[:,0], errs[:,1]))
        return med_err, len(sightings)
    finally:
        globals()["VIDEO_OFFSET_S"] = old_offset

results = []
for off in np.arange(-8.0, 1.0, 0.2):
    med_err, n_obs = calibration_error_for_offset(off, SHELTER_SIGHTINGS, FOCAL_PX)
    if med_err is not None:
        results.append((off, med_err, n_obs))
        print(f"offset={off:+.2f}s  median error={med_err:6.1f}px  (n={n_obs})")

results.sort(key=lambda x: x[1])
best_offset = results[0][0]
print(f"\nBest offset: {best_offset:+.2f}s (error {results[0][1]:.1f}px), "
      f"was VIDEO_OFFSET_S={VIDEO_OFFSET_S}")

offset=-8.00s  median error=   8.1px  (n=57)

offset=-7.80s  median error=   8.1px  (n=57)

offset=-7.60s  median error=   8.1px  (n=57)

offset=-7.40s  median error=   8.9px  (n=61)

offset=-7.20s  median error=   8.6px  (n=61)

offset=-7.00s  median error=   8.2px  (n=61)

offset=-6.80s  median error=   8.1px  (n=61)

offset=-6.60s  median error=   8.1px  (n=61)

offset=-6.40s  median error=   8.0px  (n=61)

offset=-6.20s  median error=   8.1px  (n=61)

offset=-6.00s  median error=   8.0px  (n=61)

offset=-5.80s  median error=   8.9px  (n=65)

offset=-5.60s  median error=   8.9px  (n=65)

offset=-5.40s  median error=   8.9px  (n=65)

offset=-5.20s  median error=   8.9px  (n=65)

offset=-5.00s  median error=   8.9px  (n=65)

offset=-4.80s  median error=   8.9px  (n=65)

offset=-4.60s  median error=   8.2px  (n=65)

offset=-4.40s  median error=   8.0px  (n=65)

offset=-4.20s  median error=   8.0px  (n=65)

offset=-4.00s  median error=   8.9px  (n=69)

offset=-3.80s  median error=   8.9px  (n=69)

offset=-3.60s  median error=   8.8px  (n=69)

offset=-3.40s  median error=   8.5px  (n=69)

offset=-3.20s  median error=   8.2px  (n=69)

offset=-3.00s  median error=   8.0px  (n=69)

offset=-2.80s  median error=   7.9px  (n=69)

offset=-2.60s  median error=   7.9px  (n=69)

offset=-2.40s  median error=   8.0px  (n=73)

offset=-2.20s  median error=   8.0px  (n=73)

offset=-2.00s  median error=   8.0px  (n=73)

offset=-1.80s  median error=   7.9px  (n=73)

offset=-1.60s  median error=   7.8px  (n=73)

offset=-1.40s  median error=   7.8px  (n=73)

offset=-1.20s  median error=   7.4px  (n=73)

offset=-1.00s  median error=   7.2px  (n=73)

offset=-0.80s  median error=   7.7px  (n=77)

offset=-0.60s  median error=   7.3px  (n=77)

offset=-0.40s  median error=   6.7px  (n=77)

offset=-0.20s  median error=   6.5px  (n=77)

offset=+0.00s  median error=   6.2px  (n=77)

offset=+0.20s  median error=   6.1px  (n=77)

offset=+0.40s  median error=   5.8px  (n=77)

offset=+0.60s  median error=   5.8px  (n=77)

offset=+0.80s  median error=   5.3px  (n=77)

 Best offset: +0.80s (error 5.3px), was VIDEO_OFFSET_S=0.8


## 10. Apply to tracks

In [ ]:
# Project all detections to lat/lon using the same pitch/height cutoffs, plus a max-distance sanity filter.

if not os.path.exists(TRACKS_XML):
    print(f"{TRACKS_XML} not found.")
elif "PITCH_BY_FRAME" not in dir():
    print("calibrate pitch/focal (requires ≥5 occultation observations)..")
else:
    with open("/kaggle/working/ref_point.json") as f:
        _ref_on_disk = json.load(f)
    assert abs(_ref_on_disk["REF_LAT"] - REF_LAT) < 1e-9 and abs(_ref_on_disk["REF_LNG"] - REF_LNG) < 1e-9, \
        "The REF_LAT/REF_LNG values in memory have diverged from ref_point.json!"

    MAX_GROUND_DISTANCE_M = 150.0

    tree = ET.parse(TRACKS_XML)
    root = tree.getroot()
    rows = []
    n_rejected_distance = n_rejected_height = n_rejected_pitch = 0
    for track in root.findall(".//track"):
        track_id = track.get("id")
        for box in track.findall("box"):
            if box.get("outside") == "1":
                continue
            frame = int(box.get("frame"))
            cx = (float(box.get("xtl")) + float(box.get("xbr"))) / 2
            cy = (float(box.get("ytl")) + float(box.get("ybr"))) / 2
            lat, lng, h, yaw = get_drone_state(frame)
            if h < MIN_HEIGHT_FOR_TRUST_M:
                n_rejected_height += 1
                continue
            pitch, gap = get_pitch_for_frame(frame)
            if pitch < MIN_PITCH_FOR_TRUST_DEG:
                n_rejected_pitch += 1
                continue
            de, dn = latlng_to_local_m(lat, lng, REF_LAT, REF_LNG)
            proj = project_ray_to_height(de, dn, h, yaw, pitch, FOCAL_PX, cx, cy, target_height=0)
            if proj is None:
                continue
            east, north = proj
            dist_from_drone = np.hypot(east - de, north - dn)
            if dist_from_drone > MAX_GROUND_DISTANCE_M:
                n_rejected_distance += 1
                continue
            lat_out, lon_out = local_m_to_latlng(east, north, REF_LAT, REF_LNG)
            rows.append({
                "track_id": track_id, "frame": frame, "time_s": frame / VIDEO_FPS,
                "east": east, "north": north, "lat": lat_out, "lon": lon_out, "gap": gap,
            })
    df_geo = pd.DataFrame(rows)
    df_geo.to_csv("/kaggle/working/chicken_positions_v4.csv", index=False)
    print(f" Geotagged {len(df_geo)} detection, {df_geo['track_id'].nunique() if len(df_geo) else 0} tracks")
    print(f" Discarded based on height (<{MIN_HEIGHT_FOR_TRUST_M} m): {n_rejected_height}")
    print(f" Rejected based on the pitch (<{MIN_PITCH_FOR_TRUST_DEG}°): {n_rejected_pitch}")
    print(f" Discarded by distance (>{MAX_GROUND_DISTANCE_M} m from the drone): {n_rejected_distance}")
    if len(df_geo):
        print(f"gap: median={df_geo['gap'].median():.0f} frames, 95-th perc.={df_geo['gap'].quantile(0.95):.0f} frames")
        print(f"east range: {df_geo['east'].min():.1f}…{df_geo['east'].max():.1f} m")
        print(f"north range: {df_geo['north'].min():.1f}…{df_geo['north'].max():.1f} m")
        if df_geo["east"].abs().max() > 1000 or df_geo["north"].abs().max() > 1000:
            print("The east/north range exceeds 1 km even after filtering.")

Geotagged 119110 detection, 139 tracks

Discarded based on height (<8.0 m): 18751

Rejected based on the pitch (<25.0°): 82936

Discarded by distance (>150.0 m from the drone): 0

gap: median=724 frames, 95-th perc.=3959 frames

east range: -48.9…2.9 m

north range: -37.4…14.0 m


## 11. Keyframe check

In [ ]:
# Check if box.keyframe separates real vs. interpolated detections. Expected to be 100% here by construction - speed-based segmentation (below) is the actual filter.

tree = ET.parse(TRACKS_XML)
root = tree.getroot()
sample_box = root.find(".//track/box")
print("Attributes box:", sample_box.attrib)

if "keyframe" in sample_box.attrib:
    total = kf = 0
    for track in root.findall(".//track"):
        for box in track.findall("box"):
            if box.get("outside") == "1":
                continue
            total += 1
            if box.get("keyframe") == "1":
                kf += 1
    print(f"Real keyframe: {kf} from {total} ({kf/total*100:.1f}%)")
else:
    print("There is no 'keyframe' attribute. Capture velocity-based interpolation.")

Attributes box: {'frame': '3', 'xtl': '816.5', 'ytl': '512.1', 'xbr': '860.0', 'ybr': '555.0', 'outside': '0', 'occluded': '0', 'keyframe': '1'}
Real keyframe: 220797 from 220797 (100.0%)

## 12. Smoothing

In [ ]:
# Median-filter (east, north) per track to remove single-frame noise (worst at shallow pitch) while keeping real movement.

from scipy.signal import medfilt

SMOOTH_WINDOW = 5   # odd number of frames - tune to your detection density

df_geo = df_geo.sort_values(["track_id", "frame"]).reset_index(drop=True)

smoothed_east, smoothed_north = [], []
for tid, grp in df_geo.groupby("track_id"):
    e, n = grp["east"].values, grp["north"].values
    if len(e) >= SMOOTH_WINDOW:
        e_s, n_s = medfilt(e, kernel_size=SMOOTH_WINDOW), medfilt(n, kernel_size=SMOOTH_WINDOW)
    else:
        e_s, n_s = e, n   # too short to smooth - leave as is
    smoothed_east.extend(e_s)
    smoothed_north.extend(n_s)

df_geo["east_raw"], df_geo["north_raw"] = df_geo["east"], df_geo["north"]
df_geo["east"], df_geo["north"] = smoothed_east, smoothed_north

print(f"✓ Trajectories smoothed using a median filter (window = {SMOOTH_WINDOW} frames)")
print(f"  original coordinates saved in east_raw/north_raw")

 Trajectories smoothed using a median filter (window = 5 frames)
  original coordinates saved in east_raw/north_raw

## 14. Speed-based segmentation

In [ ]:
# Split tracks where implied speed exceeds physical chicken speed. Catches occlusion jumps, ID switches, calibration noise - cause doesn't matter, only feasibility.

MAX_CHICKEN_SPEED_MS = 3.0   # walking ~0.3-1 m/s, running ~3-4 m/s - some margin included

df_geo = df_geo.sort_values(["track_id", "frame"]).reset_index(drop=True)

segment_ids = []
seg_counter = 0
prev_track = prev_row = None
jump_speeds = []

for idx, row in df_geo.iterrows():
    if row["track_id"] != prev_track:
        seg_counter += 1
    elif prev_row is not None:
        dt = row["time_s"] - prev_row["time_s"]
        dist = np.hypot(row["east"] - prev_row["east"], row["north"] - prev_row["north"])
        if dt > 0:
            speed = dist / dt
            if speed > MAX_CHICKEN_SPEED_MS:
                seg_counter += 1
                jump_speeds.append((row["track_id"], prev_row["frame"], row["frame"], speed, dist, dt))
    segment_ids.append(seg_counter)
    prev_track, prev_row = row["track_id"], row

df_geo["segment_id"] = segment_ids

n_tracks, n_segments = df_geo["track_id"].nunique(), df_geo["segment_id"].nunique()
print(f"Tracks: {n_tracks} → segments after cutting, by speed: {n_segments}")
print(f"Breaks (implied speed > {MAX_CHICKEN_SPEED_MS} m/s): {len(jump_speeds)}")

if jump_speeds:
    jump_speeds.sort(key=lambda x: -x[3])
    print(f"\n{'track_id':>10} {'frame1':>8} {'frame2':>8} {'speed,m/s':>13} {'distance,m':>12} {'dt,s':>6}")
    for tid, f1, f2, sp, d, dt in jump_speeds[:20]:
        print(f"{tid:>10} {f1:>8} {f2:>8} {sp:>13.1f} {d:>12.1f} {dt:>6.2f}")
    print(f"... (the 20 worst ones are shown {len(jump_speeds)})")
    print("\nIf dt here is significantly larger than 1/VIDEO_FPS, it represents a detection failure")
    print("(occlusion), rather than the adjacent frame: the segment break here is correct.")

df_geo.to_csv("/kaggle/working/chicken_positions_v7.csv", index=False)


Tracks: 139 → segments after cutting, by speed: 20026

Breaks (implied speed > 3.0 m/s): 19887

  track_id   frame1   frame2     speed,m/s   distance,m   dt,s

      1165    23732    23733         282.7          4.7   0.02
      1165    23727    23728         276.2          4.6   0.02
      1165    23745    23746         267.2          4.5   0.02
      1165    23739    23740         267.0          4.5   0.02
      1153    23732    23733         214.5          3.6   0.02
      1153    23727    23728         213.1          3.6   0.02
      1153    23739    23740         200.6          3.3   0.02
      1153    23745    23746         198.7          3.3   0.02
      1165    23720    23721         189.4          3.2   0.02
      1153    23720    23721         141.1          2.4   0.02
      1165    23750    23751         138.4          2.3   0.02
       705     3951     3952         133.4          2.2   0.02
       835     3951     3952         131.5          2.2   0.02
       950     6291     6292         111.9          1.9   0.02
       967     6165     6166         111.8          1.9   0.02
       967     6171     6172         110.5          1.8   0.02
       950     6171     6172         109.6          1.8   0.02
       705     3945     3946         108.7          1.8   0.02
       967     6291     6292         106.8          1.8   0.02
       950     6165     6166         106.6          1.8   0.02
... (the 20 worst ones are shown 19887)

If dt here is significantly larger than 1/VIDEO_FPS, it represents a detection failure
(occlusion), rather than the adjacent frame: the segment break here is correct.

## 14. Final map

In [ ]:
# Final static map: high-confidence points only, split into speed-consistent segments.
GAP_THRESHOLD_FRAMES = 700

df_geo = pd.read_csv("/kaggle/working/chicken_positions_v7.csv")
high_conf = df_geo[df_geo["gap"] <= GAP_THRESHOLD_FRAMES].copy()

fig, ax = plt.subplots(figsize=(8, 8))

with open("/kaggle/working/ref_point.json") as f:
    _ref = json.load(f)
east_t, north_t = latlng_to_local_m(telemetry["lat"].values, telemetry["lng"].values, _ref["REF_LAT"], _ref["REF_LNG"])

ax.plot(east_t, north_t, "-", color="lightgray", linewidth=1, label="drone track")
for seg_id, grp in high_conf.groupby("segment_id"):
    ax.plot(grp["east"], grp["north"], "o-", markersize=3, linewidth=1, alpha=0.7)

ax.set_xlabel("East, m"); ax.set_ylabel("North, m"); ax.set_aspect("equal")
ax.set_title(f"Chickens, nothing but high confidence (gap≤{GAP_THRESHOLD_FRAMES}), segments by speed")
ax.legend(); ax.grid(alpha=0.3)

# Guard before plotting: fail loudly if scale is way off, instead of silently saving a bad plot.
span = max(east_t.max()-east_t.min(), north_t.max()-north_t.min())
assert span < 1000, f"The scale of the drone's track is suspiciously large: {span:.0f} m - need to check REF_LAT/REF_LNG!"

plt.tight_layout(); plt.savefig("/kaggle/working/map_high_confidence.png", dpi=110); plt.show()

![](../../data/images/Screenshot%202026-09-15%20at%2014.41.54.png)

## 15. Interactive timeline widget

In [ ]:
# Interactive timeline: drone + chosen track over time. HTML/canvas/JS (matplotlib widgets unreliable on Kaggle), runs client-side.

import json as _json
from IPython.display import HTML

def _track_payload(df, track_id, id_col="segment_id", max_points=4000):
    grp = df[df["track_id"].astype(str) == str(track_id)].sort_values("frame")
    if len(grp) == 0:
        print(f"⚠ For track_id={track_id!r} not found in df "
              f"(types: df['track_id'].dtype={df['track_id'].dtype}). "
              f"Available ID (first 10): {sorted(df['track_id'].astype(str).unique())[:10]}")
        return None
    if len(grp) > max_points:
        step = len(grp) // max_points + 1
        grp = grp.iloc[::step]
    has_seg = id_col in grp.columns
    pts, prev_seg = [], None
    for _, r in grp.iterrows():
        seg = r[id_col] if has_seg else None
        pts.append({
            "t": round(float(r["time_s"]), 3),
            "e": round(float(r["east"]), 2),
            "n": round(float(r["north"]), 2),
            "brk": bool(has_seg and prev_seg is not None and seg != prev_seg),
        })
        prev_seg = seg
    return pts

def _drone_payload(telemetry, ref_lat, ref_lng, max_points=3000):
    t = telemetry["time_s"].values
    e, n = latlng_to_local_m(telemetry["lat"].values, telemetry["lng"].values, ref_lat, ref_lng)
    if len(t) > max_points:
        step = len(t) // max_points + 1
        t, e, n = t[::step], e[::step], n[::step]
    return [{"t": round(float(tt), 2), "e": round(float(ee), 2), "n": round(float(nn), 2)}
            for tt, ee, nn in zip(t, e, n)]

def chicken_time_slider(df, telemetry, ref_lat, ref_lng, track_ids=None,
                         id_col="segment_id", max_tracks=25, width=760, height=760):
    """
    df - high_conf or df_geo (needs track_id, frame, time_s, east, north[, segment_id]).
    track_ids=None -> the max_tracks longest tracks by point count.
    track_ids=[...] -> only the listed track_id values.
    """
    if track_ids is None:
        counts = df.groupby("track_id").size().sort_values(ascending=False)
        track_ids = counts.head(max_tracks).index.tolist()

    tracks_data = {}
    for tid in track_ids:
        payload = _track_payload(df, tid, id_col=id_col)
        if payload:
            tracks_data[str(tid)] = payload
    if not tracks_data:
        print("⚠ No tracks were found for the provided track_ids..")
        return None

    drone_pts = _drone_payload(telemetry, ref_lat, ref_lng)
    all_e = [p["e"] for p in drone_pts] + [pt["e"] for pts in tracks_data.values() for pt in pts]
    all_n = [p["n"] for p in drone_pts] + [pt["n"] for pts in tracks_data.values() for pt in pts]
    pad = 5
    bounds = {"eMin": min(all_e)-pad, "eMax": max(all_e)+pad, "nMin": min(all_n)-pad, "nMax": max(all_n)+pad}
    max_t = max(p["t"] for p in drone_pts)

    data_json = _json.dumps({"drone": drone_pts, "tracks": tracks_data, "bounds": bounds, "maxT": max_t})
    uid = f"tw_{abs(hash(tuple(str(t) for t in track_ids))) % 100000}"

    template = """
    <div style="font-family: sans-serif;">
      <div style="margin-bottom:6px;">
        Chicken: <select id="sel___UID__"></select> &nbsp;&nbsp; <button id="play___UID__">▶ Play</button>
      </div>
      <div style="margin-bottom:6px;">
        Video time: <span id="time___UID__">0.0</span> s (frame ~<span id="frame___UID__">0</span>)
      </div>
      <input id="slider___UID__" type="range" min="0" max="1000" value="0" style="width:__WIDTH__px;">
      <br>
      <canvas id="canvas___UID__" width="__WIDTH__" height="__HEIGHT__"
              style="border:1px solid #999; margin-top:8px;"></canvas>
    </div>
    <script>
    (function() {
        const data = __DATA_JSON__;
        const VIDEO_FPS = __FPS__;
        const W = __WIDTH__, H = __HEIGHT__;
        const eMin = data.bounds.eMin, eMax = data.bounds.eMax;
        const nMin = data.bounds.nMin, nMax = data.bounds.nMax;

        const canvas = document.getElementById("canvas___UID__");
        const ctx = canvas.getContext("2d");
        const slider = document.getElementById("slider___UID__");
        const sel = document.getElementById("sel___UID__");
        const timeLabel = document.getElementById("time___UID__");
        const frameLabel = document.getElementById("frame___UID__");
        const playBtn = document.getElementById("play___UID__");

        Object.keys(data.tracks).forEach(function(tid) {
            const opt = document.createElement("option");
            opt.value = tid;
            opt.textContent = "track " + tid + " (" + data.tracks[tid].length + " т.)";
            sel.appendChild(opt);
        });

        const N_STEPS = 1000;
        slider.max = N_STEPS;

        function toPx(e, n) {
            const x = (e - eMin) / (eMax - eMin) * (W - 40) + 30;
            const y = H - ((n - nMin) / (nMax - nMin) * (H - 40) + 20);
            return [x, y];
        }

        function interpDrone(t) {
            const pts = data.drone;
            if (t <= pts[0].t) return pts[0];
            if (t >= pts[pts.length-1].t) return pts[pts.length-1];
            let lo = 0, hi = pts.length - 1;
            while (hi - lo > 1) { const mid = (lo + hi) >> 1; if (pts[mid].t <= t) lo = mid; else hi = mid; }
            const a = pts[lo], b = pts[hi];
            const f = (t - a.t) / (b.t - a.t || 1);
            return { e: a.e + (b.e - a.e) * f, n: a.n + (b.n - a.n) * f };
        }

        function draw() {
            const frac = slider.value / N_STEPS;
            const t = frac * data.maxT;
            timeLabel.textContent = t.toFixed(1);
            frameLabel.textContent = Math.round(t * VIDEO_FPS);

            ctx.clearRect(0, 0, W, H);

            ctx.strokeStyle = "#ccc"; ctx.lineWidth = 1;
            ctx.beginPath();
            data.drone.forEach(function(p, i) {
                const xy = toPx(p.e, p.n);
                if (i === 0) ctx.moveTo(xy[0], xy[1]); else ctx.lineTo(xy[0], xy[1]);
            });
            ctx.stroke();

            const tid = sel.value;
            if (tid && data.tracks[tid]) {
                const pts = data.tracks[tid];

                ctx.strokeStyle = "#f5b7c8"; ctx.lineWidth = 1.5;
                ctx.beginPath();
                let started = false;
                pts.forEach(function(p) {
                    const xy = toPx(p.e, p.n);
                    if (p.brk || !started) { ctx.moveTo(xy[0], xy[1]); started = true; }
                    else ctx.lineTo(xy[0], xy[1]);
                });
                ctx.stroke();

                ctx.strokeStyle = "#d6336c"; ctx.lineWidth = 2.5;
                ctx.beginPath();
                started = false;
                let lastPt = null;
                pts.forEach(function(p) {
                    if (p.t > t) return;
                    const xy = toPx(p.e, p.n);
                    if (p.brk || !started) { ctx.moveTo(xy[0], xy[1]); started = true; }
                    else ctx.lineTo(xy[0], xy[1]);
                    lastPt = p;
                });
                ctx.stroke();

                if (lastPt) {
                    const xy = toPx(lastPt.e, lastPt.n);
                    ctx.fillStyle = "#d6336c";
                    ctx.beginPath(); ctx.arc(xy[0], xy[1], 6, 0, 2*Math.PI); ctx.fill();
                }
            }

            const dp = interpDrone(t);
            const dxy = toPx(dp.e, dp.n);
            ctx.fillStyle = "#1971c2";
            ctx.beginPath(); ctx.arc(dxy[0], dxy[1], 7, 0, 2*Math.PI); ctx.fill();
            ctx.strokeStyle = "#fff"; ctx.lineWidth = 1.5; ctx.stroke();
        }

        slider.addEventListener("input", draw);
        sel.addEventListener("change", draw);
        if (sel.options.length > 0) { sel.selectedIndex = 0; }

        let playing = false, playTimer = null;
        playBtn.addEventListener("click", function() {
            playing = !playing;
            playBtn.textContent = playing ? "⏸ Pause" : "▶ Play";
            if (playing) {
                playTimer = setInterval(function() {
                    let v = parseInt(slider.value) + 3;
                    if (v > N_STEPS) v = 0;
                    slider.value = v;
                    draw();
                }, 40);
            } else {
                clearInterval(playTimer);
            }
        });

        draw();
    })();
    </script>
    """
    html = (template.replace("__UID__", uid).replace("__WIDTH__", str(width)).replace("__HEIGHT__", str(height))
            .replace("__FPS__", str(VIDEO_FPS)).replace("__DATA_JSON__", data_json))
    return HTML(html)

chicken_time_slider(high_conf, telemetry, REF_LAT, REF_LNG)                    # 25 longest tracks
chicken_time_slider(high_conf, telemetry, REF_LAT, REF_LNG, track_ids=[1011])   # a specific chicken

## 16. Map + video slideshow widget

In [ ]:
# Map + video thumbnail slideshow, shared time slider - visual sanity check of calibration against original footage.

import cv2, base64
import re as _re
from pathlib import Path as _Path

_FRAME_FILES_CACHE = {}

def _get_frame_files(frames_dir):
    if frames_dir not in _FRAME_FILES_CACHE:
        def _frame_number(path):
            m = _re.search(r'(\d+)', path.stem)
            return int(m.group(1)) if m else 0
        exts = {'.jpg', '.jpeg', '.png'}
        files = sorted([p for p in _Path(frames_dir).rglob("*") if p.suffix.lower() in exts], key=_frame_number)
        if not files:
            raise FileNotFoundError(f"In {frames_dir} not found .jpg/.jpeg/.png")
        _FRAME_FILES_CACHE[frames_dir] = files
        print(f" Frames found in {frames_dir}: {len(files)}")
    return _FRAME_FILES_CACHE[frames_dir]

def _drone_payload_video_time(telemetry, ref_lat, ref_lng, max_points=3000):
    video_t = telemetry["time_s"].values - VIDEO_OFFSET_S   # telemetry time -> video time
    e, n = latlng_to_local_m(telemetry["lat"].values, telemetry["lng"].values, ref_lat, ref_lng)
    order = np.argsort(video_t)
    video_t, e, n = video_t[order], e[order], n[order]
    if len(video_t) > max_points:
        step = len(video_t)//max_points + 1
        video_t, e, n = video_t[::step], e[::step], n[::step]
    return [{"t": round(float(t),3), "e": round(float(ee),2), "n": round(float(nn),2)}
            for t, ee, nn in zip(video_t, e, n)]

def _get_track_frame_range(xml_root, track_id):
    track_id = str(track_id)
    for track in xml_root.findall(".//track"):
        if track.get("id") == track_id:
            frames = [int(b.get("frame")) for b in track.findall("box") if b.get("outside") != "1"]
            if frames:
                return min(frames), max(frames)
    return None

def _collect_boxes_in_range(xml_root, frame_lo, frame_hi):
    boxes_by_frame = {}
    for track in xml_root.findall(".//track"):
        tid = track.get("id")
        for b in track.findall("box"):
            if b.get("outside") == "1":
                continue
            f = int(b.get("frame"))
            if f < frame_lo or f > frame_hi:
                continue
            boxes_by_frame.setdefault(f, []).append(
                (tid, float(b.get("xtl")), float(b.get("ytl")), float(b.get("xbr")), float(b.get("ybr")))
            )
    return boxes_by_frame

def _make_thumb_b64(frame_num, boxes, highlight_track_id, thumb_width, frame_files, quality=70):
    if frame_num < 0 or frame_num >= len(frame_files):
        return None
    img = cv2.imread(str(frame_files[frame_num]))
    if img is None:
        return None
    H, W = img.shape[:2]
    scale = thumb_width / W
    img = cv2.resize(img, (thumb_width, max(1, int(H*scale))))
    for tid, xtl, ytl, xbr, ybr in boxes:
        x1, y1, x2, y2 = int(xtl*scale), int(ytl*scale), int(xbr*scale), int(ybr*scale)
        is_sel = str(tid) == str(highlight_track_id)
        color = (0, 0, 255) if is_sel else (160, 160, 160)   # BGR
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 3 if is_sel else 1)
        if is_sel:
            cv2.putText(img, f"id {tid}", (x1, max(0, y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return base64.b64encode(buf).decode() if ok else None

def _track_payload(df, track_id, id_col="segment_id", max_points=4000):
    grp = df[df["track_id"].astype(str) == str(track_id)].sort_values("frame")
    if len(grp) == 0:
        print(f"There are no points for track_id={track_id!r} in the provided df "
              f"(types: df['track_id'].dtype={df['track_id'].dtype}). "
              f"Available ID (first 10): {sorted(df['track_id'].astype(str).unique())[:10]}")
        return None
    if len(grp) > max_points:
        step = len(grp) // max_points + 1
        grp = grp.iloc[::step]
    has_seg = id_col in grp.columns
    pts, prev_seg = [], None
    for _, r in grp.iterrows():
        seg = r[id_col] if has_seg else None
        pts.append({
            "t": round(float(r["time_s"]), 3), "e": round(float(r["east"]), 2), "n": round(float(r["north"]), 2),
            "brk": bool(has_seg and prev_seg is not None and seg != prev_seg),
        })
        prev_seg = seg
    return pts

def map_video_widget(df, telemetry, ref_lat, ref_lng, track_id, xml_root, id_col="segment_id",
                      max_frames=120, thumb_width=480, map_width=480, map_height=480, frames_dir=None):
    """
    df - high_conf or df_geo. xml_root - an already-parsed root from ET.parse(TRACKS_XML)
    (reuse `root` from the tracks-application cell). frames_dir defaults to the global FRAMES_DIR.
    """
    frames_dir = frames_dir or FRAMES_DIR
    frame_files = _get_frame_files(frames_dir)

    frame_range = _get_track_frame_range(xml_root, track_id)
    if frame_range is None:
        print(f"Track {track_id} not found in XML.")
        return None
    frame_lo, frame_hi = frame_range
    pad = int(0.5 * VIDEO_FPS)
    frame_lo, frame_hi = max(0, frame_lo - pad), min(len(frame_files) - 1, frame_hi + pad)

    boxes_by_frame = _collect_boxes_in_range(xml_root, frame_lo, frame_hi)
    frames_with_track = sorted(f for f, bs in boxes_by_frame.items() if any(str(tid) == str(track_id) for tid, *_ in bs))
    if not frames_with_track:
        print(f"For track {track_id} there are not boxes in range {frame_lo}-{frame_hi}.")
        return None

    step = max(1, len(frames_with_track) // max_frames)
    sampled_frames = frames_with_track[::step]

    print(f"Preparing {len(sampled_frames)} from the frame range {frame_lo}-{frame_hi}")
    thumbs = {}
    for f in sampled_frames:
        b64 = _make_thumb_b64(f, boxes_by_frame.get(f, []), track_id, thumb_width, frame_files)
        if b64:
            thumbs[f] = b64
    if not thumbs:
        print("Failed to read any frame files check frames_dir")
        return None
    print(f" {len(thumbs)} ready (~{sum(len(v) for v in thumbs.values())//1024} kb base64)")

    track_path = _track_payload(df, track_id, id_col=id_col) or []
    drone_pts = _drone_payload_video_time(telemetry, ref_lat, ref_lng)

    all_e = [p["e"] for p in drone_pts] + [p["e"] for p in track_path]
    all_n = [p["n"] for p in drone_pts] + [p["n"] for p in track_path]
    pad_m = 5
    bounds = {"eMin": min(all_e)-pad_m, "eMax": max(all_e)+pad_m, "nMin": min(all_n)-pad_m, "nMax": max(all_n)+pad_m}

    sorted_frames = sorted(thumbs.keys())
    min_t, max_t = sorted_frames[0]/VIDEO_FPS, sorted_frames[-1]/VIDEO_FPS

    data_json = _json.dumps({
        "drone": drone_pts, "track": track_path,
        "thumbs": {str(f): thumbs[f] for f in sorted_frames}, "thumbFrames": sorted_frames,
        "bounds": bounds, "minT": min_t, "maxT": max_t, "fps": VIDEO_FPS,
    })
    uid = f"mv_{abs(hash(str(track_id))) % 100000}"

    template = """
    <div style="font-family: sans-serif;">
      <div style="margin-bottom:6px;">
        Chicken <b>__TID__</b> - video time: <span id="time___UID__">0.0</span> with
        (frame ~<span id="frame___UID__">0</span>)
      </div>
      <div style="display:flex; gap:14px; align-items:flex-start;">
        <canvas id="canvas___UID__" width="__MAPW__" height="__MAPH__" style="border:1px solid #999;"></canvas>
        <img id="video___UID__" width="__MAPW__" style="border:1px solid #999; background:#eee;">
      </div>
      <input id="slider___UID__" type="range" min="0" max="1000" value="0" style="width:__TOTALW__px; margin-top:8px;">
      <br>
      <button id="play___UID__">▶ Play</button> &nbsp;&nbsp;
      Video correction:
      <input id="offset___UID__" type="number" step="0.1" value="0" style="width:60px;">с
      <button id="reset_offset___UID__" style="margin-left:4px;">reset</button>
    </div>
    <script>
    (function() {
        const data = __DATA_JSON__;
        const W = __MAPW__, H = __MAPH__;
        const eMin = data.bounds.eMin, eMax = data.bounds.eMax;
        const nMin = data.bounds.nMin, nMax = data.bounds.nMax;

        const canvas = document.getElementById("canvas___UID__");
        const ctx = canvas.getContext("2d");
        const img = document.getElementById("video___UID__");
        const slider = document.getElementById("slider___UID__");
        const timeLabel = document.getElementById("time___UID__");
        const frameLabel = document.getElementById("frame___UID__");
        const playBtn = document.getElementById("play___UID__");
        const offsetInput = document.getElementById("offset___UID__");
        const resetOffsetBtn = document.getElementById("reset_offset___UID__");

        const N_STEPS = 1000;
        slider.max = N_STEPS;

        function toPx(e, n) {
            const x = (e - eMin) / (eMax - eMin) * (W - 40) + 30;
            const y = H - ((n - nMin) / (nMax - nMin) * (H - 40) + 20);
            return [x, y];
        }

        function interpDrone(t) {
            const pts = data.drone;
            if (t <= pts[0].t) return pts[0];
            if (t >= pts[pts.length-1].t) return pts[pts.length-1];
            let lo = 0, hi = pts.length - 1;
            while (hi - lo > 1) { const mid = (lo + hi) >> 1; if (pts[mid].t <= t) lo = mid; else hi = mid; }
            const a = pts[lo], b = pts[hi];
            const f = (t - a.t) / (b.t - a.t || 1);
            return { e: a.e + (b.e - a.e) * f, n: a.n + (b.n - a.n) * f };
        }

        function nearestThumbFrame(t) {
            const frames = data.thumbFrames;
            const target = t * data.fps;
            let lo = 0, hi = frames.length - 1;
            if (target <= frames[0]) return frames[0];
            if (target >= frames[hi]) return frames[hi];
            while (hi - lo > 1) { const mid = (lo + hi) >> 1; if (frames[mid] <= target) lo = mid; else hi = mid; }
            return (Math.abs(frames[lo]-target) <= Math.abs(frames[hi]-target)) ? frames[lo] : frames[hi];
        }

        function draw() {
            const frac = slider.value / N_STEPS;
            const t = data.minT + frac * (data.maxT - data.minT);
            timeLabel.textContent = t.toFixed(1);
            frameLabel.textContent = Math.round(t * data.fps);

            ctx.clearRect(0, 0, W, H);

            ctx.strokeStyle = "#ccc"; ctx.lineWidth = 1;
            ctx.beginPath();
            data.drone.forEach(function(p, i) {
                const xy = toPx(p.e, p.n);
                if (i === 0) ctx.moveTo(xy[0], xy[1]); else ctx.lineTo(xy[0], xy[1]);
            });
            ctx.stroke();

            const pts = data.track;
            ctx.strokeStyle = "#f5b7c8"; ctx.lineWidth = 1.5;
            ctx.beginPath();
            let started = false;
            pts.forEach(function(p) {
                const xy = toPx(p.e, p.n);
                if (p.brk || !started) { ctx.moveTo(xy[0], xy[1]); started = true; }
                else ctx.lineTo(xy[0], xy[1]);
            });
            ctx.stroke();

            ctx.strokeStyle = "#d6336c"; ctx.lineWidth = 2.5;
            ctx.beginPath();
            started = false;
            let lastPt = null;
            pts.forEach(function(p) {
                if (p.t > t) return;
                const xy = toPx(p.e, p.n);
                if (p.brk || !started) { ctx.moveTo(xy[0], xy[1]); started = true; }
                else ctx.lineTo(xy[0], xy[1]);
                lastPt = p;
            });
            ctx.stroke();
            if (lastPt) {
                const xy = toPx(lastPt.e, lastPt.n);
                ctx.fillStyle = "#d6336c";
                ctx.beginPath(); ctx.arc(xy[0], xy[1], 6, 0, 2*Math.PI); ctx.fill();
            }

            const dp = interpDrone(t);
            const dxy = toPx(dp.e, dp.n);
            ctx.fillStyle = "#1971c2";
            ctx.beginPath(); ctx.arc(dxy[0], dxy[1], 7, 0, 2*Math.PI); ctx.fill();
            ctx.strokeStyle = "#fff"; ctx.lineWidth = 1.5; ctx.stroke();

            const shownT = t + parseFloat(offsetInput.value || "0");
            const fnum = nearestThumbFrame(shownT);
            img.src = "data:image/jpeg;base64," + data.thumbs[String(fnum)];
        }

        slider.addEventListener("input", draw);
        offsetInput.addEventListener("input", draw);
        resetOffsetBtn.addEventListener("click", function() { offsetInput.value = 0; draw(); });

        let playing = false, playTimer = null;
        playBtn.addEventListener("click", function() {
            playing = !playing;
            playBtn.textContent = playing ? "⏸ Pause" : "▶ Play";
            if (playing) {
                playTimer = setInterval(function() {
                    let v = parseInt(slider.value) + 3;
                    if (v > N_STEPS) v = 0;
                    slider.value = v;
                    draw();
                }, 60);
            } else {
                clearInterval(playTimer);
            }
        });

        draw();
    })();
    </script>
    """
    html = (template.replace("__UID__", uid).replace("__TID__", str(track_id))
            .replace("__MAPW__", str(map_width)).replace("__MAPH__", str(map_height))
            .replace("__TOTALW__", str(map_width*2 + 14)).replace("__DATA_JSON__", data_json))
    return HTML(html)

map_video_widget(high_conf, telemetry, REF_LAT, REF_LNG, track_id="843", xml_root=root)